# Plotly figures for the web

> **Install for this page:** export data with `pip install scope-profiler`;
> render it in a web project with
> `npm install @scope-profiler/plotly plotly.js-dist-min`.

`@scope-profiler/plotly` is the browser-facing companion to
`scope-profiler export plot-data --format json`. It builds plain Plotly figure
specifications from the exported JSON, without importing Plotly itself. That
makes it suitable for static sites and any JavaScript framework.

In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
from importlib.util import find_spec
from pathlib import Path

from IPython.display import Image, display

WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-guide-"))

# The mock solver: a `setup` phase, then three `timestep`s with nested regions.
# Work per rank shrinks with the rank count (imperfectly), communication grows.
SOLVER = """
import sys
import time

from mpi4py import MPI

from scope_profiler import ProfileManager

size = MPI.COMM_WORLD.Get_size()


def scaled(seconds, serial_fraction=0.15):
    return seconds * (serial_fraction + (1.0 - serial_fraction) / size)


ProfileManager.setup(file_path=sys.argv[1])

with ProfileManager.region("setup"):
    time.sleep(scaled(0.060))

for step in range(3):
    with ProfileManager.region("timestep"):
        with ProfileManager.region("assemble"):
            time.sleep(scaled(0.025))
        with ProfileManager.region("solve"):
            time.sleep(scaled(0.060))
        with ProfileManager.region("halo_exchange"):
            time.sleep(0.004 * size)
        if step % 2 == 1:
            with ProfileManager.region("io"):
                time.sleep(0.012)

ProfileManager.finalize(verbose=False)
"""
(WORKDIR / "solver.py").write_text(SOLVER)

mpirun = shutil.which("mpirun")
if mpirun is None or find_spec("mpi4py") is None:
    raise RuntimeError(
        "This page needs mpirun and mpi4py to run the solver on 1, 2 and 4 ranks"
    )

env = {
    **os.environ,
    "HDF5_USE_FILE_LOCKING": "FALSE",
    "OMPI_ALLOW_RUN_AS_ROOT": "1",
    "OMPI_ALLOW_RUN_AS_ROOT_CONFIRM": "1",
}
for ranks in (1, 2, 4):
    subprocess.run(
        [
            mpirun,
            "--oversubscribe",
            "-n",
            str(ranks),
            sys.executable,
            "solver.py",
            f"run_{ranks}.h5",
        ],
        cwd=WORKDIR,
        env=env,
        check=True,
    )

print(sorted(path.name for path in WORKDIR.glob("*.h5")))

In [ ]:
def sp(command):
    """Run `scope-profiler <command>` in the working directory and print its output."""
    print(f"$ scope-profiler {command}")
    result = subprocess.run(
        [sys.executable, "-m", "scope_profiler", *shlex.split(command)],
        cwd=WORKDIR,
        capture_output=True,
        text=True,
        check=False,
    )
    print((result.stdout + result.stderr).rstrip())
    if result.returncode:
        raise RuntimeError(
            f"scope-profiler {command} failed with code {result.returncode}"
        )


def show(path):
    """Display a PNG written under the working directory."""
    display(Image(filename=WORKDIR / path))


def listing(directory):
    """Print the files under a directory of the working directory, with sizes."""
    for path in sorted((WORKDIR / directory).rglob("*")):
        if path.is_file():
            print(f"{path.relative_to(WORKDIR)}  ({path.stat().st_size} bytes)")

## Install

Install the figure builders plus the Plotly bundle your application wants to
use:

```sh
npm install @scope-profiler/plotly plotly.js-dist-min
```

Then fetch a JSON payload and render it with your own Plotly instance:

```js
import Plotly from "plotly.js-dist-min";
import { buildGanttFigure, renderFigure } from "@scope-profiler/plotly";

const payload = await fetch("/figures/gantt_data.json").then((response) => response.json());
const figure = buildGanttFigure(payload);
await renderFigure(Plotly, document.querySelector("#gantt"), figure);
```

The `gantt_data.json` fetched there is written by the exporter. The cell below
produces it from a small mock solver run (at 1, 2 and 4 MPI ranks) and shows
the start of the payload:

In [ ]:
sp("export plot-data run_2.h5 -o figures --format json --plots gantt durations")

import json

payload = json.loads((WORKDIR / "figures" / "gantt_data.json").read_text())
print(json.dumps({**payload, "intervals": payload["intervals"][:2]}, indent=2))

## Live examples

The figures below are built by the published npm package in this page. Each
uses the same shape of payload that `scope-profiler export plot-data --format
json` writes.

In [ ]:
from IPython.display import HTML

# The page's own copy of the npm package (docs/public/plotly) and Plotly from a CDN draw
# these figures in the browser, so they show on the built site, not in a plain Jupyter session.
LIVE_EXAMPLES = r"""
<div style="display: grid; gap: 1.5rem; margin: 1rem 0;">
  <div><strong>Gantt — a lane per region and rank</strong><div id="scope-plotly-gantt" style="min-height: 290px;"></div></div>
  <div><strong>Durations — compare regions between runs</strong><div id="scope-plotly-durations" style="min-height: 350px;"></div></div>
  <div><strong>Flame — nested calls</strong><div id="scope-plotly-flame" style="min-height: 440px;"></div></div>
  <div><strong>Speedup — region scaling</strong><div id="scope-plotly-speedup" style="min-height: 390px;"></div></div>
  <div><strong>Duration time series — performance drift</strong><div id="scope-plotly-timeseries" style="min-height: 390px;"></div></div>
  <div><strong>Histogram — call-duration distribution</strong><div id="scope-plotly-histogram" style="min-height: 380px;"></div></div>
  <div><strong>Rank heatmap — duration by region and rank</strong><div id="scope-plotly-heatmap" style="min-height: 340px;"></div></div>
  <div><strong>Imbalance — per-rank duration</strong><div id="scope-plotly-imbalance" style="min-height: 390px;"></div></div>
  <div><strong>Density — timeline occupancy per region</strong><div id="scope-plotly-density" style="min-height: 300px;"></div></div>
  <div><strong>Call graph — who calls whom</strong><div id="scope-plotly-callgraph" style="min-height: 340px;"></div></div>
  <div><strong>Region summary — the slowest regions in a run</strong><div id="scope-plotly-summary" style="min-height: 320px;"></div></div>
</div>
<script src="https://cdn.plot.ly/plotly-3.7.0.min.js"></script>
<script type="module">
  const packageUrl = "../../plotly/index.js";
  const { buildGanttFigure, buildDurationsFigure, buildFlameFigure, buildSpeedupFigure, buildDurationTimeseriesFigure, buildHistogramFigure, buildRankHeatmapFigure, buildImbalanceFigure, buildFigure, renderFigure } = await import(packageUrl);
  const render = (id, figure) => renderFigure(window.Plotly, document.getElementById(id), figure);
  await render("scope-plotly-gantt", buildGanttFigure({
    colors: { setup: "#2a78d6", solve: "#eb6834", exchange: "#1baf7a" },
    intervals: [
      { file: "run", rank: 0, region: "setup", start_seconds: 0, end_seconds: 0.7 },
      { file: "run", rank: 0, region: "solve", start_seconds: 0.8, end_seconds: 2.8 },
      { file: "run", rank: 0, region: "exchange", start_seconds: 2.9, end_seconds: 3.4 },
      { file: "run", rank: 1, region: "setup", start_seconds: 0, end_seconds: 0.6 },
      { file: "run", rank: 1, region: "solve", start_seconds: 0.7, end_seconds: 2.5 },
      { file: "run", rank: 1, region: "exchange", start_seconds: 2.6, end_seconds: 3.3 },
    ],
  }));
  await render("scope-plotly-durations", buildDurationsFigure({
    metrics: ["total"], colors: { "1 rank": "#2a78d6", "4 ranks": "#eb6834" },
    bars: [
      { file: "1 rank", region: "setup", metric: "total", value_seconds: 0.7 },
      { file: "1 rank", region: "solve", metric: "total", value_seconds: 5.2 },
      { file: "1 rank", region: "exchange", metric: "total", value_seconds: 1.4 },
      { file: "4 ranks", region: "setup", metric: "total", value_seconds: 0.8 },
      { file: "4 ranks", region: "solve", metric: "total", value_seconds: 1.6 },
      { file: "4 ranks", region: "exchange", metric: "total", value_seconds: 0.5 },
    ],
  }));
  await render("scope-plotly-flame", buildFlameFigure({
    colors: { timestep: "#2a78d6", assemble: "#1baf7a", solve: "#eb6834" },
    calls: [
      { file: "run", rank: 0, call_id: 1, parent_call_id: null, region: "timestep", start_seconds: 0, end_seconds: 4, inclusive_duration_seconds: 4 },
      { file: "run", rank: 0, call_id: 2, parent_call_id: 1, region: "assemble", start_seconds: 0.2, end_seconds: 1.1, inclusive_duration_seconds: 0.9 },
      { file: "run", rank: 0, call_id: 3, parent_call_id: 1, region: "solve", start_seconds: 1.2, end_seconds: 3.8, inclusive_duration_seconds: 2.6 },
    ],
  }));
  await render("scope-plotly-speedup", buildSpeedupFigure({
    colors: { solve: "#eb6834", exchange: "#1baf7a" },
    options: { x_field: "num_ranks", x_label: "MPI ranks", baseline: 1 },
    points: [
      { region: "solve", num_ranks: 1, speedup: 1 }, { region: "solve", num_ranks: 2, speedup: 1.9 }, { region: "solve", num_ranks: 4, speedup: 3.5 },
      { region: "exchange", num_ranks: 1, speedup: 1 }, { region: "exchange", num_ranks: 2, speedup: 1.7 }, { region: "exchange", num_ranks: 4, speedup: 2.6 },
    ],
  }));
  await render("scope-plotly-timeseries", buildDurationTimeseriesFigure({
    colors: { solve: "#eb6834", exchange: "#1baf7a" },
    points: [
      { region: "solve", call_index: 0, time_seconds: 1, mean_duration_seconds: 0.42, min_duration_seconds: 0.39, max_duration_seconds: 0.45 },
      { region: "solve", call_index: 1, time_seconds: 2, mean_duration_seconds: 0.47, min_duration_seconds: 0.43, max_duration_seconds: 0.50 },
      { region: "solve", call_index: 2, time_seconds: 3, mean_duration_seconds: 0.54, min_duration_seconds: 0.50, max_duration_seconds: 0.58 },
      { region: "exchange", call_index: 0, time_seconds: 1, mean_duration_seconds: 0.11, min_duration_seconds: 0.09, max_duration_seconds: 0.13 },
      { region: "exchange", call_index: 1, time_seconds: 2, mean_duration_seconds: 0.13, min_duration_seconds: 0.11, max_duration_seconds: 0.15 },
      { region: "exchange", call_index: 2, time_seconds: 3, mean_duration_seconds: 0.16, min_duration_seconds: 0.14, max_duration_seconds: 0.18 },
    ],
  }));
  await render("scope-plotly-histogram", buildHistogramFigure({
    colors: { solve: "#eb6834", exchange: "#1baf7a" },
    bins: [
      { region: "solve", bin_low_seconds: 0.3, bin_center_seconds: 0.35, bin_high_seconds: 0.4, count: 2 },
      { region: "solve", bin_low_seconds: 0.4, bin_center_seconds: 0.45, bin_high_seconds: 0.5, count: 7 },
      { region: "solve", bin_low_seconds: 0.5, bin_center_seconds: 0.55, bin_high_seconds: 0.6, count: 4 },
      { region: "exchange", bin_low_seconds: 0.1, bin_center_seconds: 0.15, bin_high_seconds: 0.2, count: 9 },
      { region: "exchange", bin_low_seconds: 0.2, bin_center_seconds: 0.25, bin_high_seconds: 0.3, count: 3 },
    ],
  }));
  await render("scope-plotly-heatmap", buildRankHeatmapFigure({
    points: [
      { rank: 0, region: "setup", total_duration_seconds: 0.7 }, { rank: 0, region: "solve", total_duration_seconds: 2.0 }, { rank: 0, region: "exchange", total_duration_seconds: 0.4 },
      { rank: 1, region: "setup", total_duration_seconds: 0.8 }, { rank: 1, region: "solve", total_duration_seconds: 2.4 }, { rank: 1, region: "exchange", total_duration_seconds: 0.5 },
      { rank: 2, region: "setup", total_duration_seconds: 0.7 }, { rank: 2, region: "solve", total_duration_seconds: 2.1 }, { rank: 2, region: "exchange", total_duration_seconds: 0.4 },
    ],
  }));
  await render("scope-plotly-imbalance", buildImbalanceFigure({
    metric: "total", colors: { solve: "#eb6834", exchange: "#1baf7a" },
    points: [
      { region: "solve", rank: 0, value_seconds: 2.0, mean_over_ranks_seconds: 2.2 }, { region: "solve", rank: 1, value_seconds: 2.6, mean_over_ranks_seconds: 2.2 }, { region: "solve", rank: 2, value_seconds: 2.0, mean_over_ranks_seconds: 2.2 },
      { region: "exchange", rank: 0, value_seconds: 0.4, mean_over_ranks_seconds: 0.5 }, { region: "exchange", rank: 1, value_seconds: 0.6, mean_over_ranks_seconds: 0.5 }, { region: "exchange", rank: 2, value_seconds: 0.5, mean_over_ranks_seconds: 0.5 },
    ],
  }));
  await render("scope-plotly-density", buildFigure({
    plot: "density",
    points: [
      { file: "run", region: "solve", bin_start_seconds: 0, bin_end_seconds: 1, occupied_seconds: 0.35 },
      { file: "run", region: "solve", bin_start_seconds: 1, bin_end_seconds: 2, occupied_seconds: 0.92 },
      { file: "run", region: "solve", bin_start_seconds: 2, bin_end_seconds: 3, occupied_seconds: 0.81 },
      { file: "run", region: "exchange", bin_start_seconds: 0, bin_end_seconds: 1, occupied_seconds: 0.05 },
      { file: "run", region: "exchange", bin_start_seconds: 1, bin_end_seconds: 2, occupied_seconds: 0.12 },
      { file: "run", region: "exchange", bin_start_seconds: 2, bin_end_seconds: 3, occupied_seconds: 0.44 },
    ],
  }));
  await render("scope-plotly-callgraph", buildFigure({
    plot: "callgraph",
    regions: [
      { name: "timestep", depth: 0, total_duration: 4.0 },
      { name: "assemble", depth: 1, total_duration: 0.9 },
      { name: "solve", depth: 1, total_duration: 2.6 },
      { name: "precondition", depth: 2, total_duration: 1.1 },
    ],
    edges: [
      { parent: "timestep", child: "assemble" }, { parent: "timestep", child: "solve" },
      { parent: "solve", child: "precondition" },
    ],
  }));
  await render("scope-plotly-summary", buildFigure({
    plot: "region_statistics",
    files: [{ label: "run", region_statistics: {
      setup: { count: 1, total_duration_seconds: 0.7 },
      solve: { count: 24, total_duration_seconds: 5.2 },
      exchange: { count: 24, total_duration_seconds: 1.4 },
      diagnostics: { count: 6, total_duration_seconds: 0.3 },
    } }],
  }, { topN: 4 }));
</script>
"""

display(HTML(LIVE_EXAMPLES))

## One call for any payload

`buildFigure` reads the `plot` kind stamped on every plot-data document and
picks the builder itself, so a page that renders whatever the profiler wrote
does not need to know which chart it is getting:

```js
import Plotly from "plotly.js-dist-min";
import { buildFigure, renderFigure } from "@scope-profiler/plotly";

for (const name of ["gantt", "durations", "rank_heatmap"]) {
  const payload = await fetch(`/figures/${name}_data.json`).then((response) => response.json());
  await renderFigure(Plotly, document.querySelector(`#${name}`), buildFigure(payload));
}
```

It throws on a document that is not `scope-profiler-plot-data`, and on one
whose `format_version` is newer than the installed package understands. For
JSON written by a scope-profiler old enough to predate the envelope, it falls
back to `inferPlotKind(payload)`, which reads the shape of the payload
instead; pass `{ plot: "gantt" }` to settle the case by hand.

## Builders

| Builder | Input payload | Use case |
| --- | --- | --- |
| `buildGanttFigure(payload, { laneBy })` | `gantt_data.json` | Timeline of every call, a lane per region and rank (`laneBy: "rank"` for one row per rank) |
| `buildFlameFigure(payload)` | `flame_chart_data.json` (or `flame_graph_data.json`) | Nested call hierarchy |
| `buildDurationsFigure(payload, { metric })` | `durations_data.json` | Per-region duration bars |
| `buildSpeedupFigure(payload)` | `speedup_data.json` | Speedup curve |
| `buildDurationTimeseriesFigure(payload)` | `duration_timeseries_data.json` | Duration drift over time |
| `buildHistogramFigure(payload)` | `histogram_data.json` | Call-duration distribution |
| `buildRankHeatmapFigure(payload)` | `rank_heatmap_data.json` | Rank × region duration matrix |
| `buildImbalanceFigure(payload)` | `imbalance_data.json` | Per-rank duration imbalance |
| `buildDensityFigure(payload)` | `timeline_density_data.json` | Binned timeline occupancy, for traces too large to draw call by call |
| `buildCallgraphFigure(payload)` | `callgraph_data.json` | Sankey call graph, from either callgraph shape |
| `buildRegionSummaryFigure(payload, { metric, topN, files, orientation })` | `region_statistics.json` | The slowest regions of a run, ranked |
| `buildComparisonFigure(payload, { files, metric })` | `region_statistics.json` | Every run side by side, over the regions they all recorded; `{ comparison: "percent" \| "absolute" }` draws a single delta bar instead, for exactly two runs |
| `buildLikwidFigure(payload)` | `likwid_data.json` | One LIKWID hardware-counter metric |
| `buildRooflineFigure(payload)` | `roofline_data.json` | Per-region LIKWID roofline placement |
| `buildWeakScalingFigure(payload)` | `weak_scaling_data.json` | Weak-scaling curve |
| `buildScalingEfficiencyFigure(payload)` | `scaling_efficiency_data.json` | Strong-scaling parallel-efficiency curve |
| `buildWeakScalingEfficiencyFigure(payload)` | `weak_scaling_efficiency_data.json` | Weak-scaling efficiency curve |
| `buildFigure(payload)` | any of the above | Dispatch on the document's own `plot` kind |

The exporter writes these files. Here every kind that needs no LIKWID data is
exported from the three mock runs, with the `plot` stamp and the row list of
each payload:

In [ ]:
sp(
    "export plot-data run_1.h5 run_2.h5 run_4.h5 -o all_kinds --format json "
    "--plots gantt flame_chart durations speedup timeseries histogram rank_heatmap "
    "imbalance density callgraph weak_scaling scaling_efficiency weak_scaling_efficiency"
)

for path in sorted((WORKDIR / "all_kinds").glob("*_data.json")):
    document = json.loads(path.read_text())
    rows = next(key for key, value in document.items() if isinstance(value, list))
    print(f"{path.name:36} plot={document['plot']!r:28} rows under {rows!r}")

Use `renderFigure(Plotly, element, figure)` as a convenience, or pass
`figure.data` and `figure.layout` to `Plotly.newPlot` directly.

Both efficiency curves store their y column as `efficiency`, so which reading
a payload gets is decided by the document's own `plot` field, not by its rows:
`buildFigure` gets it right on its own, and a direct call needs the right
builder. They measure against different ideals. `scaling_efficiency` is for a
*strong*-scaling study, where the problem size is fixed and the ideal is a
speedup proportional to the rank count. `weak_scaling_efficiency` is for a
study that grows the problem with the machine, where every run does the same
work per rank and the ideal is constant runtime.

## Themes

Every builder paints its chrome - text, gridlines, hover surface, the dashed
ideal lines - from a theme. The default, `auto`, leaves the text colour unset
and draws gridlines in a half-transparent grey that reads on any background,
so a figure inherits whatever the host page does.

A page that knows which theme it is in gets better contrast by saying so:

```js
import { setTheme, buildDurationsFigure } from "@scope-profiler/plotly";

// Once per theme change, then rebuild the figures on the page.
setTheme(document.documentElement.dataset.theme === "dark" ? "dark" : "light");

// Or per call, which wins over the default.
buildDurationsFigure(payload, { theme: "dark" });

// Or with your own tokens, over any of the three.
buildDurationsFigure(payload, { theme: { text: "#f5f5f5", grid: "#333" } });
```

Plotly bakes colours into the layout, so a theme change means rebuilding the
figures, not restyling them in place. The categorical series palette is not
themed: those eight hues are chosen to read on both backgrounds.

Rebuilding means knowing which containers hold a figure and what each was built
from. `@scope-profiler/plotly/dashboard` keeps that bookkeeping, along with a
filter syntax for the `filterRegion` predicate - see [Deploying a profiling dashboard](/scope-profiler/guide/dashboard/).

## Payloads with more than one run

`export plot-data` accepts several profiles at once, and every row of the
resulting payload names the run it came from. The builders keep those runs
apart rather than pooling them: the gantt, density and rank-heatmap charts give
each run its own lanes, while the histogram, imbalance and duration time series
give each run its own trace, named `run / region`. A region keeps its colour
across runs - what changes between runs is the marker symbol on a line, or the
bar pattern on a histogram - so the same region stays recognisable while the
runs stay distinguishable. Single-run payloads are unaffected: series are named
by region alone, as before.

The exported payloads from the three runs above show the run name on every row:

In [ ]:
for name, rows in [
    ("gantt", "intervals"),
    ("histogram", "bins"),
    ("imbalance", "points"),
]:
    document = json.loads((WORKDIR / "all_kinds" / f"{name}_data.json").read_text())
    print(f"{name}_data.json: runs =", sorted({row["file"] for row in document[rows]}))